In [2]:
!pip install -q transformers==4.36.2 tokenizers==0.15.0 sentencepiece==0.1.99

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.8/126.8 kB 6.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 70.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 128.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 109.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 49.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.6.0 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.36.2 which is incompatible.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [1]:
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

processor = TrOCRProcessor.from_pretrained('microsoft/trocr-base-printed', use_fast=False)
model = VisionEncoderDecoderModel.from_pretrained('microsoft/trocr-base-printed')
model = model.to(device)

model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id           = processor.tokenizer.pad_token_id
model.config.vocab_size             = model.config.decoder.vocab_size

print('Model loaded successfully!')

/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


...

merges.txt: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

Some weights of VisionEncoderDecoderModel were not initialized from the model checkpoint at microsoft/trocr-base-printed and are newly initialized: ['encoder.pooler.dense.weight', 'encoder.pooler.dense.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


generation_config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded successfully!


In [2]:
urdu_chars = list('ابپتٹثجچحخدڈرڑزژسشصضطظعغفقکگلمنوہھءیےآأؤئ')
num_added = processor.tokenizer.add_tokens(urdu_chars)
print(f'Added {num_added} new Urdu tokens to tokenizer')

model.decoder.resize_token_embeddings(len(processor.tokenizer))
model.config.vocab_size = len(processor.tokenizer)

print(f'New vocab size: {len(processor.tokenizer)}')
print('Model resized successfully!')

Added 41 new Urdu tokens to tokenizer
New vocab size: 50306
Model resized successfully!


In [3]:
!git clone https://github.com/fatimasajid31/urdu-ocr-codesaviours-si26-fatima.git repo_data

Cloning into 'repo_data'...
remote: Enumerating objects: 471, done.
remote: Counting objects: 100% (471/471), done.
remote: Compressing objects: 100% (465/465), done.
remote: Total 471 (delta 80), reused 138 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (471/471), 2.25 MiB | 6.16 MiB/s, done.
Resolving deltas: 100% (80/80), done.


In [4]:
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import pandas as pd

class UrduOCRDataset(Dataset):
    def __init__(self, csv_path, processor, base_folder=''):
        self.data = pd.read_csv(csv_path)
        self.processor = processor
        self.base_folder = base_folder
        print(f'Dataset loaded: {len(self.data)} samples')

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        image_path = f"{self.base_folder}/{row['image']}" if self.base_folder else row['image']
        image = Image.open(image_path).convert('RGB')

        encoding = self.processor(image, return_tensors='pt').pixel_values.squeeze()
        pixel_values = encoding

        labels = self.processor.tokenizer(
            row['text'],
            padding='max_length',
            max_length=128
        ).input_ids
        labels = torch.tensor(labels)

        return {'pixel_values': pixel_values, 'labels': labels}

print('UrduOCRDataset class defined successfully!')

UrduOCRDataset class defined successfully!


In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split

csv_path = 'repo_data/data/labels.csv'
base_folder = 'repo_data'

df = pd.read_csv(csv_path)
print(f'Total samples: {len(df)}')

train_df, test_df = train_test_split(df, test_size=0.15, random_state=42)
train_df.to_csv('train_labels.csv', index=False)
test_df.to_csv('test_labels.csv', index=False)

print(f'Train samples: {len(train_df)}')
print(f'Test samples: {len(test_df)}')

train_dataset = UrduOCRDataset(csv_path='train_labels.csv', processor=processor, base_folder=base_folder)
test_dataset  = UrduOCRDataset(csv_path='test_labels.csv',  processor=processor, base_folder=base_folder)

print('Datasets created successfully!')

Total samples: 224
Train samples: 190
Test samples: 34
Dataset loaded: 190 samples
Dataset loaded: 34 samples
Datasets created successfully!


In [6]:
from torch.utils.data import DataLoader
from transformers import AdamW

train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=2)

optimizer = AdamW(model.parameters(), lr=5e-5)

print(f'Training batches per epoch: {len(train_loader)}')
print('Ready to train!')

/usr/local/lib/python3.12/dist-packages/transformers/optimization.py:429: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


Training batches per epoch: 95
Ready to train!


In [7]:
import gc

num_epochs = 3

for epoch in range(num_epochs):
    model.train()
    total_loss = 0

    print(f'\nEpoch {epoch + 1}/{num_epochs}')
    print('-' * 30)

    for batch_idx, batch in enumerate(train_loader):
        pixel_values = batch['pixel_values'].to(device)
        labels        = batch['labels'].to(device)

        labels[labels == processor.tokenizer.pad_token_id] = -100

        outputs = model(pixel_values=pixel_values, labels=labels)
        loss    = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        batch_loss = loss.item()

        del pixel_values, labels, outputs, loss
        gc.collect()

        if batch_idx % 10 == 0:
            print(f'  Batch {batch_idx}/{len(train_loader)} | Loss: {batch_loss:.4f}')

    avg_loss = total_loss / len(train_loader)
    print(f'Epoch {epoch + 1} complete | Average Loss: {avg_loss:.4f}')

print('\nTraining complete!')


Epoch 1/3
------------------------------
  Batch 0/95 | Loss: 10.2330
  Batch 10/95 | Loss: 5.6861
  Batch 20/95 | Loss: 3.7947
  Batch 30/95 | Loss: 3.9824
  Batch 40/95 | Loss: 1.7663
  Batch 50/95 | Loss: 2.9988
  Batch 60/95 | Loss: 3.0758
  Batch 70/95 | Loss: 2.8834
  Batch 80/95 | Loss: 3.0921
  Batch 90/95 | Loss: 3.3124
Epoch 1 complete | Average Loss: 3.8669

Epoch 2/3
------------------------------
  Batch 0/95 | Loss: 3.0059
  Batch 10/95 | Loss: 3.2982
  Batch 20/95 | Loss: 3.2184
  Batch 30/95 | Loss: 3.0973
  Batch 40/95 | Loss: 2.9699
  Batch 50/95 | Loss: 3.2262
  Batch 60/95 | Loss: 2.6784
  Batch 70/95 | Loss: 3.1544
  Batch 80/95 | Loss: 3.0177
  Batch 90/95 | Loss: 3.0867
Epoch 2 complete | Average Loss: 3.0069

Epoch 3/3
------------------------------
  Batch 0/95 | Loss: 3.3700
  Batch 10/95 | Loss: 2.2557
  Batch 20/95 | Loss: 2.7863
  Batch 30/95 | Loss: 3.1220
  Batch 40/95 | Loss: 2.9521
  Batch 50/95 | Loss: 3.2969
  Batch 60/95 | Loss: 3.4052
  Batch 70/95

In [8]:
!pip install -q python-Levenshtein

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 57.4 MB/s eta 0:00:00


In [9]:
def calculate_cer(pred, actual):
    """Character Error Rate - lower is better"""
    import Levenshtein
    if len(actual) == 0:
        return 1.0
    return Levenshtein.distance(pred, actual) / len(actual)

model.eval()
predictions = []
actuals = []

with torch.no_grad():
    for batch in test_loader:
        pixel_values = batch['pixel_values'].to(device)
        labels = batch['labels']

        # FIX: proper generate parameters
        generated_ids = model.generate(
            pixel_values,
            max_length=128,
            num_beams=4,
            early_stopping=True
        )

        pred_texts = processor.batch_decode(generated_ids, skip_special_tokens=True)

        labels_clean = labels.clone()
        labels_clean[labels_clean == -100] = processor.tokenizer.pad_token_id
        actual_texts = processor.batch_decode(labels_clean, skip_special_tokens=True)

        predictions.extend(pred_texts)
        actuals.extend(actual_texts)

# Show first 5 predictions vs actual
for i in range(min(5, len(predictions))):
    print(f'Predicted: {predictions[i]}')
    print(f'Actual:    {actuals[i]}')
    print('---')

# Calculate accuracy and CER
exact_matches = sum(1 for p, a in zip(predictions, actuals) if p.strip() == a.strip())
accuracy = exact_matches / len(predictions) * 100

cers = [calculate_cer(p, a) for p, a in zip(predictions, actuals)]
avg_cer = sum(cers) / len(cers) * 100

print(f'\nExact match accuracy: {accuracy:.2f}%')
print(f'Average CER: {avg_cer:.2f}%')

/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:1518: UserWarning: You have modified the pretrained model configuration to control generation. This is a deprecated strategy to control generation and will be removed soon, in a future version. Please use and modify the model generation configuration (see https://huggingface.co/docs/transformers/generation_strategies#default-text-generation-configuration )
  warnings.warn(


Predicted:                                                                                                                               
Actual:    م ح ن ت   م ی ں  ع ظ م ت   ہ ے
---
Predicted:                                                                                                                               
Actual:    ق و م   ک ی   ت ر ق ی   ت ع ل ی م   م ی ں  ہ ے
---
Predicted:                                                                                                                               
Actual:    ک ا م ی ا ب   ل و گ   ہ م ی ش ہ   م ح ن ت ی   ہ و ت ے   ہ ی ں
---
Predicted:                                                                                                                               
Actual:    د و س ر و ں  ک ے   ک ا م   آ ن ا   ب ڑ ی   ن ی ک ی   ہ ے
---
Predicted: 777777777777777777777777777777777777777777777777777777777777777777777777777777777777777777777777777777777777777777777777777777
Actual:    2
---

Exact match accuracy: 0.00%
Average

In [10]:
model.eval()
print('=== Model Evaluation on Test Images ===')
print()

correct = 0
total   = 0
wrong_examples = []

with torch.no_grad():
    for batch in test_loader:
        pixel_values = batch['pixel_values'].to(device)
        labels        = batch['labels']

        # FIX: proper generate parameters (avoids empty/repetitive output issues)
        generated_ids = model.generate(
            pixel_values,
            max_length=128,
            num_beams=4,
            early_stopping=True
        )
        generated_text = processor.batch_decode(generated_ids, skip_special_tokens=True)

        # FIX: convert -100 back to pad_token_id before decoding actual labels
        labels_clean = labels.clone()
        labels_clean[labels_clean == -100] = processor.tokenizer.pad_token_id
        actual_text = processor.batch_decode(labels_clean, skip_special_tokens=True)

        for pred, actual in zip(generated_text, actual_text):
            total += 1
            if pred.strip() == actual.strip():
                correct += 1
            else:
                wrong_examples.append((pred, actual))

            print(f'Predicted: {pred}')
            print(f'Actual:    {actual}')
            print()

accuracy = (correct / total) * 100 if total > 0 else 0
print(f'Accuracy: {accuracy:.1f}% ({correct}/{total} correct)')

# Show 3-5 wrong examples for your Week 5 discussion notes
print('\n--- Sample wrong predictions for report ---')
for pred, actual in wrong_examples[:5]:
    print(f'Predicted: {pred}')
    print(f'Actual:    {actual}')
    print('---')

=== Model Evaluation on Test Images ===

Predicted:                                                                                                                               
Actual:    م ح ن ت   م ی ں  ع ظ م ت   ہ ے

Predicted:                                                                                                                               
Actual:    ق و م   ک ی   ت ر ق ی   ت ع ل ی م   م ی ں  ہ ے

Predicted:                                                                                                                               
Actual:    ک ا م ی ا ب   ل و گ   ہ م ی ش ہ   م ح ن ت ی   ہ و ت ے   ہ ی ں

Predicted:                                                                                                                               
Actual:    د و س ر و ں  ک ے   ک ا م   آ ن ا   ب ڑ ی   ن ی ک ی   ہ ے

Predicted: 777777777777777777777777777777777777777777777777777777777777777777777777777777777777777777777777777777777777777777777777777777
Actual:    2

Predicted: